# Shinka + coding-agent baselines — live status

Compiles the **in-flight ShinkaEvolve baseline runs** (ac1 / ac2 / erdos, one `meta_r1` arm each,
target 2400 generations) and the **short coding-agent runs** (`coding_agent_evolve/`: erdos x3,
TriMul kernel) into one place, on axes the ICL runs will later share
(**candidates tried `k`** and generations).

**Re-runnable by design**: the Shinka runs are live tmux processes writing WAL-mode SQLite;
every loader here reads read-only (`mode=ro`, WAL-aware) and re-parses from disk, so re-running
all cells in a few hours picks up the newer generations. Nothing is cached to disk.

**Not here yet**: the real ICL runs (PUCT / BoN / best / random / contrastive) live on another
server. Section 5 is a stub that auto-loads them from `src/runs/` via `results.analysis` the
moment they are copied in.

Shinka semantics reminder: **one generation = one candidate** (one parent + inspirations -> one
proposal -> one eval), unlike the ICL loop's 6x16 fan-out. So Shinka's generation axis IS its
candidate axis, and comparisons should be made at equal `k` = candidates tried.

In [ ]:
import json, os, re, glob, sqlite3, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# project root: this notebook lives in src/notebooks/
_here = os.getcwd()
ROOT = _here
while not os.path.isdir(os.path.join(ROOT, "ShinkaEvolve")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("project root with ShinkaEvolve/ not found above " + _here)
    ROOT = parent
SHINKA = os.path.join(ROOT, "ShinkaEvolve", "examples", "ttt_discover_math")
AGENT = os.path.join(ROOT, "coding_agent_evolve")

# fixed method colors (validated categorical order -- color follows the method everywhere)
C = {"shinka": "#2a78d6", "coding_agent": "#eb6834", "icl_puct": "#1baf7a",
     "bon": "#eda100", "icl_ctx": "#e87ba4", "initial": "#898781", "target": "#0b0b0b"}

# problem specs: raw metric key in Shinka's public_metrics, direction, seed score, prompt target
# (targets/initials: src/envs/ac_inequalities.py:263-265, erdos_min_overlap.py:158; eval hard-kill
#  1100 s for all three, self-budget promised to the program is 1000 s)
PROBLEMS = {
    "ac1":   dict(metric="upper_bound", maximize=False, initial=2.0,      target=1.5030,
                  shinka_dir=os.path.join(SHINKA, "ac1", "results", "ac1_qwen_meta_r1")),
    "ac2":   dict(metric="lower_bound", maximize=True,  initial=0.904893, target=0.97,
                  shinka_dir=os.path.join(SHINKA, "ac2", "results", "ac2_qwen_meta_r1")),
    "erdos": dict(metric="c5_bound",    maximize=False, initial=0.493986, target=0.3808,
                  shinka_dir=os.path.join(SHINKA, "erdos_min_overlap", "results", "erdos_qwen_meta_r1")),
}
# best published value per problem (better of TTT-Discover / AlphaEvolve)
BEST_KNOWN = {"ac1": ("TTT-Discover", 1.50287),
              "ac2": ("AlphaEvolve", 0.9610),
              "erdos": ("TTT-Discover", 0.380876)}
EVAL_KILL_S = 1100.0   # scheduler hard kill (job_time 00:18:20)
EVAL_BUDGET_S = 1000.0 # budget the prompt promises the evolved program
SHINKA_TARGET_GENS = 2400

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("root:", ROOT)
print("loaded:", dt.datetime.now().strftime("%Y-%m-%d %H:%M"))

## 1. Shinka runs — load

One row per **program tried**, ordered by wall-clock completion, `k` = 1-based candidate index.
`raw` is the problem's own metric (`upper_bound` / `lower_bound` / `c5_bound`); `score` is Shinka's
maximize-everywhere `combined_score` (= `1/upper_bound`, `lower_bound`, `1/c5_bound` respectively).
`valid` = Shinka's `correct` flag. The duplicate gen-0 row each resume re-inserts is dropped
(known artifact, IMPLEMENTATION_LOG 2026-08-10). Per-eval timing comes from the `metadata` blob
(`evaluation_seconds` = wall clock around the eval job) and from `gen_N/results/metrics.json`
(`execution_time_mean` = the compute time the program itself reports, bounded by its 1000 s budget).

In [ ]:
def load_shinka(problem: str) -> pd.DataFrame:
    d = PROBLEMS[problem]["shinka_dir"]
    con = sqlite3.connect(f"file:{os.path.join(d, 'programs.sqlite')}?mode=ro", uri=True)
    df = pd.read_sql_query(
        """SELECT id, parent_id, generation, timestamp, combined_score AS score,
                  correct, public_metrics, metadata FROM programs ORDER BY timestamp""", con)
    con.close()
    # drop the duplicate initial-program rows a resume re-inserts (keep the first gen-0 row)
    g0 = df.index[df.generation == 0]
    df = df.drop(g0[1:]).reset_index(drop=True)

    key = PROBLEMS[problem]["metric"]
    pm = df.public_metrics.apply(lambda s: json.loads(s) if s else {})
    df["raw"] = pm.apply(lambda m: m.get(key))
    meta = df.metadata.apply(lambda s: json.loads(s) if s else {})
    for col in ("evaluation_seconds", "sampling_seconds", "pipeline_seconds"):
        df[col] = meta.apply(lambda m: m.get(col))
    df["valid"] = df.correct.astype(bool) & df.raw.notna()
    df["k"] = range(1, len(df) + 1)

    # the program's own reported compute time, from each generation dir (None if the eval died
    # before writing results)
    etm = {}
    for p in glob.glob(os.path.join(d, "gen_*", "results", "metrics.json")):
        gen = int(p.split(os.sep)[-3].split("_")[1])
        try:
            etm[gen] = json.load(open(p)).get("execution_time_mean")
        except (json.JSONDecodeError, OSError):
            pass
    df["exec_time_mean"] = df.generation.map(etm)

    # best-so-far on the raw metric, over valid rows only (ffill: cummin/cummax leave NaN
    # at invalid rows)
    s = df.raw.where(df.valid)
    df["best_raw"] = (s.cummax() if PROBLEMS[problem]["maximize"] else s.cummin()).ffill()
    df["problem"] = problem
    return df

shinka = {p: load_shinka(p) for p in PROBLEMS}
pd.concat(shinka.values())[["problem", "generation", "k", "raw", "valid", "best_raw"]].groupby("problem").tail(1)

In [ ]:
# status table -- rerun to watch the runs advance
rows = []
for p, df in shinka.items():
    spec = PROBLEMS[p]
    hours = (df.timestamp.max() - df.timestamp.min()) / 3600
    rate = (len(df) - 1) / hours if hours > 0 else np.nan
    rows.append({
        "problem": p, "candidates tried": len(df),
        "of target": f"{len(df)/SHINKA_TARGET_GENS:.1%}",
        "valid": f"{df.valid.mean():.0%}",
        "metric": spec["metric"] + (" (max)" if spec["maximize"] else " (min)"),
        "initial": spec["initial"], "best": df.best_raw.iloc[-1], "target": spec["target"],
        "wall h": round(hours, 1), "gens/h": round(rate, 1),
        "ETA @ rate (days)": round((SHINKA_TARGET_GENS - len(df)) / rate / 24, 1) if rate else None,
        "last program": dt.datetime.fromtimestamp(df.timestamp.max()).strftime("%m-%d %H:%M"),
    })
status = pd.DataFrame(rows).set_index("problem")
status

## 2. Shinka — best-so-far curves

Raw problem metric vs candidates tried. Gray dashed = the seed solution's score; black dotted =
the target the prompt asks the model to beat. For ac1 and erdos **lower is better** (curves go
down); ac2 is maximize. Faint dots are the individual valid candidates — they show how much of
the budget lands near the frontier vs. scattered.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (p, df) in zip(axes, shinka.items()):
    spec = PROBLEMS[p]
    v = df[df.valid]
    ax.plot(v.k, v.raw, ".", color=C["shinka"], alpha=0.25, ms=4)
    ax.plot(df.k, df.best_raw, color=C["shinka"], lw=2, label="Shinka (meta_r1)")
    ax.axhline(spec["initial"], color=C["initial"], ls="--", lw=1, label="initial")
    ref_name, ref_val = BEST_KNOWN[p]
    ax.axhline(ref_val, color=C["target"], ls=":", lw=1, label=f"{ref_name}: {ref_val}")
    ax.set_title(f"{p} — {spec['metric']} ({'max' if spec['maximize'] else 'min'})")
    ax.set_xlabel("candidates tried (k)")
axes[0].set_ylabel("raw metric")
axes[0].legend(fontsize=8)
fig.suptitle(f"Shinka best-so-far — snapshot {dt.datetime.now():%Y-%m-%d %H:%M}", y=1.03)
fig.tight_layout()
fig.savefig("ba_shinka_curves.png", dpi=150, bbox_inches="tight")

In [ ]:
# zoomed version: the improvements live in the 3rd-5th decimal, so clip the y-axis to the
# post-warmup band and write the exact best value on the plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (p, df) in zip(axes, shinka.items()):
    spec = PROBLEMS[p]
    tail = df[df.k >= 20].best_raw.dropna()
    lo, hi = tail.min(), tail.max()
    pad = max((hi - lo) * 0.15, 1e-4 * abs(hi))
    ax.plot(df.k, df.best_raw, color=C["shinka"], lw=2)
    final_k, final_v = df.k.iloc[-1], df.best_raw.iloc[-1]
    ax.plot(final_k, final_v, "o", ms=6, color=C["shinka"])
    ax.annotate(f"best = {final_v:.8f}", xy=(final_k, final_v),
                xytext=(-10, 12 if spec["maximize"] else -16), textcoords="offset points",
                ha="right", fontsize=9, fontweight="bold", color="#0b0b0b")
    ref_name, ref_val = BEST_KNOWN[p]
    if lo - pad <= ref_val <= hi + pad:
        ax.axhline(ref_val, color=C["target"], ls=":", lw=1)
        ax.annotate(f"{ref_name}: {ref_val}", xy=(0.02, ref_val), xycoords=("axes fraction", "data"),
                    fontsize=8, color="#52514e", va="bottom")
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_title(f"{p} — {spec['metric']} ({'max' if spec['maximize'] else 'min'}), zoomed")
    ax.set_xlabel("candidates tried (k)")
    ax.ticklabel_format(useOffset=False, style="plain", axis="y")
axes[0].set_ylabel("best-so-far raw metric")
fig.suptitle("Shinka best-so-far, zoomed past the initial drop (y-axis clipped to k >= 20)", y=1.03)
fig.tight_layout()
fig.savefig("ba_shinka_curves_zoom.png", dpi=150, bbox_inches="tight")

In [ ]:
# validity over time: rolling share of candidates that evaluate as correct
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), sharey=True)
W = 20
for ax, (p, df) in zip(axes, shinka.items()):
    r = df.valid.rolling(W, min_periods=5).mean()
    ax.plot(df.k, 100 * r, color=C["shinka"], lw=1.5)
    ax.set_ylim(0, 102)
    ax.set_title(p)
    ax.set_xlabel("candidates tried (k)")
axes[0].set_ylabel(f"% valid (rolling {W})")
fig.suptitle("Shinka — share of proposals that evaluate as valid", y=1.05)
fig.tight_layout()

## 3. Eval fidelity — did candidate evaluations get their time?

Two distinct failure modes to separate (the user-facing question: *did CPU overhead / timeouts
distort results?*):

1. **Hard kills.** The scheduler kills any eval whose wall clock exceeds 1100 s
   (`job_time 00:18:20`). Kills are logged only in `evolution_run.log`
   (`"exceeded timeout ... => Gen. N"`). A kill is **not necessarily a loss**: most killed
   programs had already finished their self-budgeted ~1000 s of work and written
   `metrics.json`, so the score was captured — the process just lingered past the wall.
   The **true losses** are killed generations with no captured result (`valid == False`).
2. **CPU-overhead inflation.** The program self-reports `execution_time_mean` (its own timer,
   promised 1000 s). If wall-clock `evaluation_seconds` is much larger than the program's own
   compute time, the gap is queue/launch/contention overhead. A program whose *self-reported*
   time is far below 1000 s but that still hit the 1100 s wall was effectively starved — it did
   NOT get the compute the prompt promised it.

Note these runs are the **relaunched (2026-08-12) processes on patched code** — the earlier
premature-timeout bug (kill clock started at proposal time) is gone, and cell outputs below
verify that: no killed eval shows a near-zero wall time.

In [ ]:
KILL_RE = re.compile(r"exceeded timeout of (\S+)\. Killing\. => Gen\. (\d+)")

def eval_fidelity(problem: str) -> pd.DataFrame:
    d = PROBLEMS[problem]["shinka_dir"]
    kills = set()
    for line in open(os.path.join(d, "evolution_run.log"), errors="replace"):
        m = KILL_RE.search(line)
        if m:
            kills.add(int(m.group(2)))
    df = shinka[problem].copy()
    df["killed"] = df.generation.isin(kills)
    df["overhead_s"] = df.evaluation_seconds - df.exec_time_mean.fillna(0)
    # starved: hit the wall while its own timer says it was far from the 1000 s budget
    df["starved"] = df.killed & df.exec_time_mean.notna() & (df.exec_time_mean < 0.9 * EVAL_BUDGET_S) & df.valid
    df["lost"] = df.killed & ~df.valid
    return df

fid = {p: eval_fidelity(p) for p in PROBLEMS}
rows = []
for p, df in fid.items():
    k = df[df.killed]
    rows.append({
        "problem": p, "evals": len(df),
        "hard kills": len(k), "kill rate": f"{len(k)/len(df):.1%}",
        "killed but score captured": int((k.valid).sum()),
        "killed and LOST": int(df.lost.sum()),
        "killed while starved (<90% of budget)": int(df.starved.sum()),
        "median overhead s (all evals)": round(df.overhead_s.median(), 0),
        "p90 overhead s": round(df.overhead_s.quantile(0.9), 0),
    })
fidelity = pd.DataFrame(rows).set_index("problem")
fidelity

In [ ]:
# wall clock vs the program's own timer, kill line marked
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, (p, df) in zip(axes, fid.items()):
    ok = df[~df.killed & df.exec_time_mean.notna()]
    kl = df[df.killed & df.exec_time_mean.notna()]
    ax.plot(ok.exec_time_mean, ok.evaluation_seconds, ".", color=C["shinka"], alpha=0.4,
            ms=5, label="normal")
    ax.plot(kl.exec_time_mean, kl.evaluation_seconds, "x", color="#d03b3b", ms=6,
            label="hard-killed")
    lim = max(EVAL_KILL_S, df.evaluation_seconds.max() or 0) * 1.05
    ax.plot([0, lim], [0, lim], color=C["initial"], lw=0.8, ls="--")   # y = x: zero overhead
    ax.axhline(EVAL_KILL_S, color="#d03b3b", lw=0.8, ls=":")
    ax.axvline(EVAL_BUDGET_S, color=C["target"], lw=0.8, ls=":")
    ax.set_title(p)
    ax.set_xlabel("program-reported compute (s)")
axes[0].set_ylabel("wall-clock eval (s)")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Eval wall clock vs self-reported compute — distance above the diagonal is overhead", y=1.03)
fig.tight_layout()
fig.savefig("ba_eval_fidelity.png", dpi=150, bbox_inches="tight")

In [ ]:
# did any kill land on a program that was ahead of the frontier at the time? (impact check)
for p, df in fid.items():
    lost = df[df.lost]
    if lost.empty:
        print(f"{p}: no killed-and-lost evals — timeouts cost nothing but wall clock")
        continue
    print(f"{p}: {len(lost)} lost eval(s) at k={lost.k.tolist()} "
          f"(gen {lost.generation.tolist()}) — scores unknown by construction; "
          f"parents' scores: {[round(s, 4) if s is not None else None for s in df.set_index('id').score.reindex(lost.parent_id).tolist()]}")

## 4. Coding-agent runs (`coding_agent_evolve/`)

Three Claude runs on **erdos** (2026-08-07/08; run1 = evo-guidance prompt in ultracode,
run2 = no evo guidance in ultracode, run3 = evo guidance at high reasoning). Each logged
differently, so each gets its own loader:

| run | source of (time, score) | scored records | caveat |
|---|---|---|---|
| run1 | `results/*.json` (`c5` field) + file **mtime** | ~112 | ~13 are **downloaded published vectors** (`W_*` tags), not search output |
| run2 | `results/pool_n*/e_<score>_<move>_<n>.npz` — score in filename + mtime | ~65 accepted (plus ~2.4k screened in `landscape_*.npy`) | **warm-started from downloaded seed vectors**; its `best_solution.json` is stale — the pools kept improving 15 h after it |
| run3 | **`results/ledger.jsonl`** (`{script, seed, tag, N, c5, t}`) | ~356 | fully self-generated — the clean run for method comparison |

**TriMul: no usable data.** The two kernel-run folders (`/scratch/vicstorage/kernel_runs/`)
were deleted and never committed; only the harness and 5 untimed variants survive. TriMul
needs a re-run before any coding-agent comparison there.

run1/run2 timestamps rely on file **mtimes**, which any future `cp` without `-p` would destroy —
so the first execution of the loader cell snapshots them to `_mtime_snapshot.json` inside each
run dir, and later executions prefer the snapshot.

In [ ]:
def _mtimes(run_dir: str, patterns: list[str]) -> dict:
    """path->mtime for the run's result files; snapshotted once so mtimes survive future copies."""
    snap_path = os.path.join(run_dir, "_mtime_snapshot.json")
    snap = json.load(open(snap_path)) if os.path.exists(snap_path) else {}
    found = {}
    for pat in patterns:
        for p in glob.glob(os.path.join(run_dir, pat)):
            rel = os.path.relpath(p, run_dir)
            found[rel] = snap.get(rel, os.path.getmtime(p))
    if set(found) - set(snap):
        json.dump({**snap, **found}, open(snap_path, "w"), indent=0)
    return found

def load_agent_run1() -> pd.DataFrame:
    d = os.path.join(AGENT, "erdos", "run1")
    mt = _mtimes(d, ["results/*.json"])
    rows = []
    for rel, t in mt.items():
        if rel.endswith("_hist.json"):
            continue
        try:
            j = json.load(open(os.path.join(d, rel)))
        except (json.JSONDecodeError, OSError):
            continue
        if "c5" not in j:
            continue
        tag = j.get("tag", os.path.basename(rel))
        rows.append(dict(t=t, c5=j["c5"], tag=tag, external=str(tag).startswith("W_")))
    df = pd.DataFrame(rows).sort_values("t").reset_index(drop=True)
    df["run"] = "run1 (evo prompt, ultracode)"
    return df

def load_agent_run2() -> pd.DataFrame:
    d = os.path.join(AGENT, "erdos", "run2")
    mt = _mtimes(d, ["results/pool_n*/e_*.npz"])
    rows = []
    for rel, t in mt.items():
        m = re.match(r"e_([0-9.]+)_(\w+)_\d+\.npz", os.path.basename(rel))
        if m:
            rows.append(dict(t=t, c5=float(m.group(1)), tag=m.group(2),
                             external=m.group(2) in ("seed", "best512")))
    df = pd.DataFrame(rows).sort_values("t").reset_index(drop=True)
    df["run"] = "run2 (no evo guidance, ultracode)"
    return df

def load_agent_run3() -> pd.DataFrame:
    p = os.path.join(AGENT, "erdos", "run3", "results", "ledger.jsonl")
    df = pd.DataFrame([json.loads(l) for l in open(p)])
    df["t"] = pd.to_datetime(df["t"]).map(lambda x: x.timestamp())
    df = df.sort_values("t").reset_index(drop=True)
    df["tag"] = df["script"]
    df["external"] = False
    df["run"] = "run3 (evo prompt, high reasoning)"
    return df[["t", "c5", "tag", "external", "run"]]

agent_runs = {f.__name__[-4:]: f() for f in (load_agent_run1, load_agent_run2, load_agent_run3)}
for name, df in agent_runs.items():
    df["k"] = range(1, len(df) + 1)
    df["hours"] = (df.t - df.t.min()) / 3600
    df["best_c5"] = df.c5.cummin().ffill()
    # best-so-far excluding DIRECT downloads (W_* / seed tags). NOTE: solutions the agent
    # *polished from* a downloaded vector keep their own tags and are NOT excluded, so for
    # run1/run2 even this curve is warm-start-contaminated after the first download; run3
    # downloaded nothing and is the only clean from-scratch series.
    df["best_c5_own"] = df.c5.where(~df.external).cummin().ffill()
pd.DataFrame({n: dict(records=len(d), external=int(d.external.sum()),
                      best=d.c5.min(), best_own=d.best_c5_own.iloc[-1],
                      wall_h=round(d.hours.iloc[-1], 1))
              for n, d in agent_runs.items()}).T

In [ ]:
# best-so-far C5, saved-attempt index and wall clock; dotted = including downloaded vectors
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
shades = {"run1": "#eb6834", "run2": "#c98500", "run3": "#a63d12"}
for name, df in agent_runs.items():
    for ax, x in ((axes[0], df.k), (axes[1], df.hours)):
        ax.plot(x, df.best_c5_own, color=shades[name], lw=1.8, label=f"{df.run.iloc[0]}")
        if df.external.any():
            ax.plot(x, df.best_c5, color=shades[name], lw=1, ls=":",
                    label=f"{name} incl. downloaded vectors")
for ax, xl in ((axes[0], "saved scored attempts"), (axes[1], "wall-clock hours")):
    ax.axhline(PROBLEMS["erdos"]["target"], color=C["target"], ls=":", lw=1)
    ax.set_xlabel(xl)
    ax.set_ylim(0.3806, 0.3825)
axes[0].set_ylabel("best C5 so far (min)")
axes[0].legend(fontsize=7)
fig.suptitle("Coding-agent erdos runs — y-axis clipped to the interesting band "
             f"(initial construction is {PROBLEMS['erdos']['initial']:.3f}, far above)", y=1.03)
fig.tight_layout()

### 4b. Anatomy of run3 — how the agent structured its search

run3 is worth dissecting because it is the only clean from-scratch run AND it left a uniform
ledger. The agent wrote **15 approach scripts** (`approaches/a01..a18`), of which **12 appear in
the ledger** (the rest were diagnostics/bounds that saved no candidates), and grouped them itself
into mathematical families in `REGISTRY.md` — several explicitly *retired* with a verdict
(multistart, smoothing, the Fourier-LP bound, the cosine basis). The `seed` column in the ledger
is the optimizer's **RNG restart seed**, not an external solution. run3 *did* search the web
(its `research_log.md` documents two search sessions: the frontier table, and the
Haugland/AlphaEvolve "few unequal-width pieces" parametrisation hint that shaped its ILS moves)
— but it **imported no solution vectors**: no network code in any of its scripts, no foreign
files in `results/`, and its score trajectory descends continuously from 0.3816 with no
cliff to a published value (its final 0.3808795 is *worse* than the 0.3808586 vectors run1/run2
downloaded — the strongest evidence it never had them).

The run has a clear phase structure (times from the ledger):

1. **23:28–00:16 — explore** (`a02` multigrid local search, `a04`/`a06` u-space basin hopping):
   collapses to the **palindromic attractor 0.380933703**, reached from every seed and grid size.
2. **00:16–00:40 — diagnose + escape attempts** (`a07` phase retrieval, `a11` variable-width
   pieces, `a13` homotopy): the agent *proves* the attractor is a strict local optimum (empty
   escape null-space) and starts deforming the objective instead of the point.
3. **00:31–02:44 — exploit** (`a14` iterated local search at fine grids + `a15` polish, 166
   records): grinds 0.38093 → 0.38090. `a17` crossover tried once, no gain (all good solutions
   shared one lineage).
4. **02:45–05:29 — assumption break** (`a18` + `asym-probe`): drops the palindromic symmetry
   assumption the whole run had been built on, finds a genuinely **asymmetric family**
   (`||h - reverse(h)|| ~ 0.5`) and resumes progress to **0.380879**.

Credit for the 76 global-best improvements: `a14` 24, `a18` 22, `a15` 16, `a04` 6, rest 8.
This is qualitatively the behavior the evo-guidance prompt asked for (registry, families,
verdicts) — and notably run2, without that guidance, produced no comparable record.

In [ ]:
# per-approach summary table straight from the ledger
r3 = agent_runs["run3"].copy()
FAMILY = {"a02": "local/multigrid", "a04": "u-space global", "a06": "u-space global",
          "a12": "u-space global", "a07": "phase/pieces/cross", "a11": "phase/pieces/cross",
          "a17": "phase/pieces/cross", "a13": "homotopy",
          "a14": "palindromic ILS+polish", "a15": "palindromic ILS+polish",
          "a18": "general (asymmetric)", "asym-probe": "general (asymmetric)"}
r3["family"] = r3.tag.map(FAMILY)
r3["new_best"] = r3.c5 == r3.c5.cummin()
summary3 = (r3.groupby("family", sort=False)
              .agg(attempts=("c5", "size"), best_c5=("c5", "min"),
                   new_bests=("new_best", "sum"),
                   first_h=("hours", "min"), last_h=("hours", "max"))
              .sort_values("first_h").round({"best_c5": 7, "first_h": 1, "last_h": 1}))
summary3

In [ ]:
# timeline: every ledger attempt, colored by approach family; black step = global best
ATTRACTOR = 0.380933703   # the palindromic attractor a04 converged to from every seed
FLOOR3 = 0.3808
fam_colors = {"local/multigrid": "#eda100", "u-space global": "#2a78d6",
              "phase/pieces/cross": "#e87ba4", "homotopy": "#1baf7a",
              "palindromic ILS+polish": "#eb6834", "general (asymmetric)": "#4a3aa7"}
fig, ax = plt.subplots(figsize=(11, 4.6))
for fam, sub in r3.groupby("family", sort=False):
    ax.plot(sub.hours, sub.c5 - FLOOR3, "o", ms=4, alpha=0.65, color=fam_colors[fam], label=fam)
ax.plot(r3.hours, r3.c5.cummin() - FLOOR3, color="#0b0b0b", lw=1.4, drawstyle="steps-post",
        label="global best")
ax.axhline(ATTRACTOR - FLOOR3, color="#898781", ls="--", lw=1)
ax.annotate("palindromic attractor (proved strict local opt)", xy=(3.1, ATTRACTOR - FLOOR3),
            fontsize=8, color="#52514e", va="bottom")
t18 = r3[r3.family == "general (asymmetric)"].hours.min()
ax.axvline(t18, color="#4a3aa7", lw=0.8, ls=":")
ax.annotate("drops palindromic\nassumption", xy=(t18 + 0.05, 2.5e-3), fontsize=8,
            color="#4a3aa7")
ax.axhline(0.380876 - FLOOR3, color="#0b0b0b", ls=":", lw=1.2, label="TTT-Discover: 0.380876")
ax.set_yscale("log")
ax.set_xlabel("hours into the run")
ax.set_ylabel(f"C5 - {FLOOR3} (log)")
ax.set_title(f"Coding-agent run3, all {len(r3)} ledger attempts")
ax.legend(fontsize=7, loc="upper right", ncol=2)
fig.tight_layout()
fig.savefig("ba_run3_anatomy.png", dpi=150, bbox_inches="tight")

## 5. Cross-method comparison — erdos (the shared problem today)

Everything below compares **best C5 so far vs candidates tried**, the axis the ICL runs will
share. Read with the caveats on the tin:

- **Budget axes are not equivalent.** A Shinka "candidate" is one full LLM proposal + a 1000 s
  eval; a coding-agent "attempt" is one *saved* artifact from an agent freely running local
  optimizers (its scripts internally screen thousands of configurations per artifact). The
  wall-clock panel is the more honest comparison for the agent.
- run1/run2 exclude *directly* downloaded vectors, but their later solutions are polished
  descendants of those downloads — treat both as **warm-started**. run3 imported no external
  solution vectors (it did consult the literature — see §4b) and is the only from-scratch
  coding-agent series, hence the one to compare against Shinka/ICL.
- The gap is plotted as `C5 - 0.3808` on a log scale — linear axes squash everything below 0.382.

When the ICL runs arrive (Section 6), their erdos arms drop onto the same axes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
FLOOR = 0.3808   # prompt target; certified LB is ~0.3808268
sh = shinka["erdos"]
axes[0].plot(sh.k, sh.best_raw - FLOOR, color=C["shinka"], lw=2, label="Shinka (meta_r1)")
sh_h = (sh.timestamp - sh.timestamp.min()) / 3600
axes[1].plot(sh_h, sh.best_raw - FLOOR, color=C["shinka"], lw=2, label="Shinka (meta_r1)")
for name, df in agent_runs.items():
    axes[0].plot(df.k, df.best_c5_own - FLOOR, color=shades[name], lw=1.6, label=df.run.iloc[0])
    axes[1].plot(df.hours, df.best_c5_own - FLOOR, color=shades[name], lw=1.6, label=df.run.iloc[0])
for ax, xl in ((axes[0], "candidates tried / saved attempts"), (axes[1], "wall-clock hours")):
    ax.set_yscale("log")
    ax.set_xlabel(xl)
    ax.axhline(PROBLEMS["erdos"]["initial"] - FLOOR, color=C["initial"], ls="--", lw=1)
axes[0].set_ylabel(f"best C5 - {FLOOR} (log)")
axes[0].legend(fontsize=7)
fig.suptitle("erdos: distance to the 0.3808 target — gray dashed = initial construction "
             "(agent runs 1-2 warm-started; budget axes differ, see caveats above)", y=1.03)
fig.tight_layout()
fig.savefig("ba_erdos_compare.png", dpi=150, bbox_inches="tight")

In [ ]:
# focused variant: one representative per paradigm — Shinka (evolutionary ICL), coding-agent
# run3 (agentic, from scratch), and the ICL PUCT run (TTT-Discover's scaffold, frozen model)
import sys
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from results import analysis as icl_analysis

fig, ax = plt.subplots(figsize=(8.5, 4.6))
sh = shinka["erdos"]
ax.plot(sh.k, sh.best_raw - FLOOR, color=C["shinka"], lw=2,
        label=f"Shinka ({len(sh)}/2400 gens so far)")
r3c = agent_runs["run3"]
ax.plot(r3c.k, r3c.best_c5 - FLOOR, color="#a63d12", lw=2,
        label="coding agent run3 (356 saved attempts, 6h)")
ev = icl_analysis.load_events(os.path.join(SRC, "runs"), "erdos", "puct_s1")
best = ev.raw_score.where(ev.valid).cummin().ffill()
ax.plot(ev.k, best - FLOOR, color=C["icl_puct"], lw=2, label="ICL PUCT (2400 candidates)")
ax.axhline(0.380876 - FLOOR, color="#0b0b0b", ls=":", lw=1.2, label="TTT-Discover: 0.380876")
ax.axhline(PROBLEMS["erdos"]["initial"] - FLOOR, color=C["initial"], ls="--", lw=1, label="initial")
ax.set_yscale("log")
ax.set_xlabel("candidates tried / saved attempts")
ax.set_ylabel(f"best C5 - {FLOOR} (log)")
ax.set_title("erdos, one line per paradigm — budget units differ for the agent (see §5 caveats)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("ba_erdos_shinka_run3_puct.png", dpi=150, bbox_inches="tight")

## 6. ICL runs — auto-loads when the results arrive

Copy the run directories from the other server into `src/runs/<group>/<run>/` (the layout
`results.analysis` already reads: `events.jsonl`, `summary.json`, `config.json`). This cell
then picks up every erdos/ac1/ac2 group automatically and overlays best-so-far vs `k` on the
same axes as above. Until then it just says what it found.

In [ ]:
import sys
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from results import analysis

idx = analysis.load_grouped_index(os.path.join(SRC, "runs"))
icl_idx = idx[idx.problem.isin(PROBLEMS)] if not idx.empty else idx
if icl_idx.empty:
    print("No ICL runs for ac1/ac2/erdos under src/runs yet -- copy them in and rerun.")
    if not idx.empty:
        print("Groups currently present:", sorted(idx.group.unique()))
else:
    display(icl_idx[["group", "run", "arm", "problem", "seed", "candidates",
                     "best_score", "status", "host"]])
    fig, axes = plt.subplots(1, len(PROBLEMS), figsize=(14, 4))
    for ax, p in zip(axes, PROBLEMS):
        sh = shinka[p]
        ax.plot(sh.k, sh.best_raw, color=C["shinka"], lw=2, label="Shinka")
        for _, r in icl_idx[icl_idx.problem == p].iterrows():
            ev = analysis.load_events(os.path.join(SRC, "runs"), r.group, r["run"])
            if ev.empty:
                continue
            s = ev.raw_score.where(ev.valid)
            best = s.cummax() if PROBLEMS[p]["maximize"] else s.cummin()
            ax.plot(ev.k, best, lw=1.2, alpha=0.8, label=r.arm)
        ax.set_title(p)
        ax.set_xlabel("candidates tried (k)")
        ax.legend(fontsize=6)
    fig.tight_layout()

## 7. Snapshot summary

Auto-generated one-liners from the current state of the data (rerun after the Shinka runs
advance).

In [ ]:
snap = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"=== snapshot {snap} ===\n")
for p, df in shinka.items():
    spec, f = PROBLEMS[p], fid[p]
    arrow = ">" if spec["maximize"] else "<"
    print(f"[shinka/{p}] {len(df)} candidates ({len(df)/SHINKA_TARGET_GENS:.0%} of 2400), "
          f"best {spec['metric']} = {df.best_raw.iloc[-1]:.6f} "
          f"(initial {spec['initial']:.4f}, target {arrow} {spec['target']}), "
          f"{df.valid.mean():.0%} valid; {int(f.killed.sum())} hard kills of which "
          f"{int(f.lost.sum())} lost a result, {int(f.starved.sum())} starved (<90% of the "
          f"promised 1000s before the 1100s wall)")
print()
for name, df in agent_runs.items():
    warm = " [warm-started from published vectors]" if df.external.any() else " [from scratch]"
    print(f"[agent/erdos {name}] {len(df)} saved attempts over {df.hours.iloc[-1]:.1f}h, "
          f"best C5 = {df.best_c5.iloc[-1]:.10f}{warm}")
print("\n[agent/trimul] no per-attempt data survives (run folders on /scratch were deleted; "
      "needs a re-run before comparison)")